#**Practical No. 8**

**Aim:**  To implement PySpark programs for reading a CSV dataset, displaying its schema, performing filtering, grouping and aggregation, removing duplicate records, joining two datasets, and calculating average sales by product category.

In [2]:
# Install PySpark in Colab
!pip install pyspark -q

# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, sum, count


In [3]:
# 1. CREATE SPARK SESSION
spark = SparkSession.builder \
    .appName("Practical8") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


In [4]:
# 2. CREATE SAMPLE CSV DATA
sales_data = """product_id,product,category,sales,customer_id
1,Laptop,Electronics,60000,101
2,Mobile,Electronics,30000,102
3,Chair,Furniture,8000,103
4,Table,Furniture,12000,104
5,Headphones,Electronics,5000,105
6,Keyboard,Electronics,3000,106
7,Chair,Furniture,8000,103
8,Monitor,Electronics,15000,107
"""

with open("sales.csv", "w") as f:
    f.write(sales_data)


customer_data = """customer_id,customer_name,city
101,Amit,Mumbai
102,Priya,Pune
103,Rahul,Nashik
104,Neha,Delhi
105,Rohan,Mumbai
106,Sneha,Pune
107,Karan,Mumbai
"""

with open("customers.csv", "w") as f:
    f.write(customer_data)

In [5]:
# 3. READ CSV DATA
sales_df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

print("\nSALES DATA:")
sales_df.show()


SALES DATA:
+----------+----------+-----------+-----+-----------+
|product_id|   product|   category|sales|customer_id|
+----------+----------+-----------+-----+-----------+
|         1|    Laptop|Electronics|60000|        101|
|         2|    Mobile|Electronics|30000|        102|
|         3|     Chair|  Furniture| 8000|        103|
|         4|     Table|  Furniture|12000|        104|
|         5|Headphones|Electronics| 5000|        105|
|         6|  Keyboard|Electronics| 3000|        106|
|         7|     Chair|  Furniture| 8000|        103|
|         8|   Monitor|Electronics|15000|        107|
+----------+----------+-----------+-----+-----------+



In [6]:
# 4. DISPLAY SCHEMA
print("\nSCHEMA:")
sales_df.printSchema()


SCHEMA:
root
 |-- product_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sales: integer (nullable = true)
 |-- customer_id: integer (nullable = true)



In [7]:
# 5. FILTERING
print("\nFILTERING - SALES GREATER THAN 10000:")

filtered_df = sales_df.filter(
    sales_df.sales > 10000
)

filtered_df.show()


FILTERING - SALES GREATER THAN 10000:
+----------+-------+-----------+-----+-----------+
|product_id|product|   category|sales|customer_id|
+----------+-------+-----------+-----+-----------+
|         1| Laptop|Electronics|60000|        101|
|         2| Mobile|Electronics|30000|        102|
|         4|  Table|  Furniture|12000|        104|
|         8|Monitor|Electronics|15000|        107|
+----------+-------+-----------+-----+-----------+



In [8]:
# 6. GROUPING AND AGGREGATION
print("\nTOTAL SALES BY CATEGORY:")

grouped_df = sales_df.groupBy(
    "category"
).agg(
    sum("sales").alias("total_sales")
)
grouped_df.show()



TOTAL SALES BY CATEGORY:
+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|     113000|
|  Furniture|      28000|
+-----------+-----------+



In [9]:
# 7. REMOVE DUPLICATE RECORDS
print("\nDATA AFTER REMOVING DUPLICATES:")

unique_df = sales_df.dropDuplicates()

unique_df.show()


DATA AFTER REMOVING DUPLICATES:
+----------+----------+-----------+-----+-----------+
|product_id|   product|   category|sales|customer_id|
+----------+----------+-----------+-----+-----------+
|         6|  Keyboard|Electronics| 3000|        106|
|         5|Headphones|Electronics| 5000|        105|
|         2|    Mobile|Electronics|30000|        102|
|         7|     Chair|  Furniture| 8000|        103|
|         1|    Laptop|Electronics|60000|        101|
|         4|     Table|  Furniture|12000|        104|
|         3|     Chair|  Furniture| 8000|        103|
|         8|   Monitor|Electronics|15000|        107|
+----------+----------+-----------+-----+-----------+



In [10]:
# 8. READ CUSTOMER DATA
customer_df = spark.read.csv(
    "customers.csv",
    header=True,
    inferSchema=True
)
print("\nCUSTOMER DATA:")
customer_df.show()


CUSTOMER DATA:
+-----------+-------------+------+
|customer_id|customer_name|  city|
+-----------+-------------+------+
|        101|         Amit|Mumbai|
|        102|        Priya|  Pune|
|        103|        Rahul|Nashik|
|        104|         Neha| Delhi|
|        105|        Rohan|Mumbai|
|        106|        Sneha|  Pune|
|        107|        Karan|Mumbai|
+-----------+-------------+------+



In [11]:
# 9. JOIN TWO DATASETS
print("\nJOINED DATA:")

joined_df = sales_df.join(
    customer_df,
    sales_df.customer_id == customer_df.customer_id,
    "inner"
)

joined_df.select(
    sales_df.product,
    sales_df.category,
    sales_df.sales,
    customer_df.customer_name,
    customer_df.city
).show()


JOINED DATA:
+----------+-----------+-----+-------------+------+
|   product|   category|sales|customer_name|  city|
+----------+-----------+-----+-------------+------+
|    Laptop|Electronics|60000|         Amit|Mumbai|
|    Mobile|Electronics|30000|        Priya|  Pune|
|     Chair|  Furniture| 8000|        Rahul|Nashik|
|     Table|  Furniture|12000|         Neha| Delhi|
|Headphones|Electronics| 5000|        Rohan|Mumbai|
|  Keyboard|Electronics| 3000|        Sneha|  Pune|
|     Chair|  Furniture| 8000|        Rahul|Nashik|
|   Monitor|Electronics|15000|        Karan|Mumbai|
+----------+-----------+-----+-------------+------+



In [12]:
# 10. AVERAGE SALES BY PRODUCT CATEGORY
print("\nAVERAGE SALES BY CATEGORY:")

average_sales = sales_df.groupBy(
    "category"
).agg(
    avg("sales").alias("average_sales")
)

average_sales.show()


AVERAGE SALES BY CATEGORY:
+-----------+-----------------+
|   category|    average_sales|
+-----------+-----------------+
|Electronics|          22600.0|
|  Furniture|9333.333333333334|
+-----------+-----------------+



In [13]:
# 11. STOP SPARK
spark.stop()
print("\nPySpark Practical 8 Completed Successfully!")


PySpark Practical 8 Completed Successfully!
